# 03 - So sanh hieu nang cac Models

Notebook nay doc `results/metrics_comparison.csv` va truc quan hoa:
1. **Pivot table** - trung binh MAE, RMSE, MASE, MAPE theo model
2. **Box plots** - do on dinh cua MAE va MASE qua cac folds/series
3. **Bar charts Phase 1 vs Phase 2** - so sanh SARIMAX, LSTM va Prophet giua 2 phase

In [ ]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)

# cau hinh chung cho bieu do dep hon
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (12, 6)

# mac dinh mau cho tung model, nhat quan xuyen suot notebook
MODEL_PALETTE = {
    "naive": "#636EFA",
    "snaive": "#EF553B",
    "sarimax": "#00CC96",
    "lstm": "#AB63FA",
    "prophet": "#FFA15A",
}

METRIC_COLS = ["MAE", "RMSE", "MASE", "MAPE"]

print("Libraries loaded.")

## 1. Doc du lieu metrics

In [ ]:
# xac dinh duong dan tuong doi toi results/
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, ".."))
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")

metrics_path = os.path.join(RESULTS_DIR, "metrics_comparison.csv")

if not os.path.exists(metrics_path):
    raise FileNotFoundError(
        f"Khong tim thay {metrics_path}. "
        "Hay chay scripts/run_experiments.py truoc."
    )

df = pd.read_csv(metrics_path)
print(f"Loaded {len(df)} rows from metrics_comparison.csv")
print(f"Models : {sorted(df['model'].unique())}")
print(f"Phases : {sorted(df['phase'].unique())}")
print(f"Folds  : {sorted(df['fold'].unique())}")
df.head()

## 2. Pivot Table - Trung binh Metrics theo Model x Phase

In [ ]:
# pivot table: trung binh toan cuc theo model + phase
pivot = (
    df.groupby(["model", "phase"])[METRIC_COLS]
    .mean()
    .round(4)
)

print("=" * 70)
print("  TRUNG BINH METRICS THEO MODEL x PHASE")
print("=" * 70)
display(pivot)

# pivot dang rong de de doc hon
pivot_wide = pivot.unstack(level="phase")
pivot_wide.columns = [
    f"{metric}_P{int(phase)}"
    for metric, phase in pivot_wide.columns
]
print("\n--- Pivot Table (wide format) ---")
display(pivot_wide)

## 3. Box Plots - Do on dinh cua MAE va MASE (Phase 1, tat ca 5 models)

In [ ]:
# chi dung Phase 1 de so sanh cong bang tat ca 5 models
df_p1 = df[df["phase"] == 1].copy()

# thu tu hien thi
model_order = [m for m in ["naive", "snaive", "sarimax", "lstm", "prophet"]
               if m in df_p1["model"].unique()]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Box plot MAE
sns.boxplot(
    data=df_p1, x="model", y="MAE",
    order=model_order,
    palette=MODEL_PALETTE,
    ax=axes[0],
    flierprops={"marker": "o", "markersize": 4, "alpha": 0.5},
)
axes[0].set_title("MAE Distribution by Model (Phase 1)", fontweight="bold")
axes[0].set_xlabel("")
axes[0].set_ylabel("MAE")

# Box plot MASE
# loai NaN truoc khi ve
df_p1_mase = df_p1.dropna(subset=["MASE"])
sns.boxplot(
    data=df_p1_mase, x="model", y="MASE",
    order=model_order,
    palette=MODEL_PALETTE,
    ax=axes[1],
    flierprops={"marker": "o", "markersize": 4, "alpha": 0.5},
)
axes[1].set_title("MASE Distribution by Model (Phase 1)", fontweight="bold")
axes[1].set_xlabel("")
axes[1].set_ylabel("MASE")
# ke duong MASE=1 lam nguong (tot hon Seasonal Naive)
axes[1].axhline(y=1.0, color="red", linestyle="--", linewidth=1, alpha=0.7,
                label="MASE = 1.0 (SNaive baseline)")
axes[1].legend(fontsize=9)

fig.suptitle("Model Stability Comparison (Phase 1, All 5 Models)",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 4. Bar Charts - Phase 1 vs Phase 2 (SARIMAX, LSTM & Prophet)

In [ ]:
# chi lay models co ca 2 phase de so sanh
phase_models = ["sarimax", "lstm", "prophet"]
df_phase_compare = df[
    (df["model"].isin(phase_models)) & (df["phase"].isin([1, 2]))
].copy()

if df_phase_compare.empty or len(df_phase_compare["phase"].unique()) < 2:
    print("[INFO] Chua co du du lieu Phase 2 de so sanh.")
    print("       Hay chay: python scripts/run_experiments.py --phase 2 --models sarimax lstm prophet")
else:
    # tinh trung binh theo model x phase
    phase_agg = (
        df_phase_compare
        .groupby(["model", "phase"])[METRIC_COLS]
        .mean()
        .reset_index()
    )
    phase_agg["phase_label"] = phase_agg["phase"].map(
        {1: "Phase 1 (Basic)", 2: "Phase 2 (Enhanced)"}
    )

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()

    phase_palette = {
        "Phase 1 (Basic)": "#636EFA",
        "Phase 2 (Enhanced)": "#00CC96",
    }

    for idx, metric in enumerate(METRIC_COLS):
        ax = axes[idx]
        sns.barplot(
            data=phase_agg,
            x="model", y=metric, hue="phase_label",
            palette=phase_palette,
            ax=ax,
            edgecolor="white",
        )
        ax.set_title(f"{metric}: Phase 1 vs Phase 2", fontweight="bold")
        ax.set_xlabel("")
        ax.set_ylabel(metric)
        ax.legend(title="", fontsize=9)

        # hien thi gia tri tren moi cot
        for container in ax.containers:
            ax.bar_label(container, fmt="%.3f", fontsize=8, padding=2)

    fig.suptitle(
        "Impact of Enhanced Features (Phase 2) on SARIMAX, LSTM & Prophet",
        fontsize=14, fontweight="bold", y=1.02,
    )
    plt.tight_layout()
    plt.show()

    # tinh phan tram thay doi
    print("\n" + "=" * 60)
    print("  % THAY DOI KHI DUNG ENHANCED FEATURES (Phase 2 vs Phase 1)")
    print("  (Gia tri am = cai thien, duong = xau di)")
    print("=" * 60)

    for model in phase_models:
        model_data = phase_agg[phase_agg["model"] == model]
        if len(model_data["phase"].unique()) < 2:
            print(f"\n  {model.upper()}: khong co du lieu Phase 2")
            continue
        p1 = model_data[model_data["phase"] == 1][METRIC_COLS].iloc[0]
        p2 = model_data[model_data["phase"] == 2][METRIC_COLS].iloc[0]

        pct_change = ((p2 - p1) / p1 * 100).round(2)
        print(f"\n  {model.upper()}:")
        for metric in METRIC_COLS:
            arrow = "\u2193" if pct_change[metric] < 0 else "\u2191"
            print(f"    {metric:>5}: {pct_change[metric]:+.2f}% {arrow}")

## 5. Heatmap - MAE trung binh theo Model x Fold (Phase 1)

In [ ]:
# heatmap cho thay model nao on dinh nhat qua cac folds
df_p1 = df[df["phase"] == 1].copy()

heatmap_data = (
    df_p1.groupby(["model", "fold"])["MAE"]
    .mean()
    .unstack(level="fold")
)

# sap xep theo thu tu model
model_order_heat = [m for m in ["naive", "snaive", "sarimax", "lstm", "prophet"]
                    if m in heatmap_data.index]
heatmap_data = heatmap_data.loc[model_order_heat]

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(
    heatmap_data,
    annot=True, fmt=".2f",
    cmap="YlOrRd",
    linewidths=0.5,
    ax=ax,
    cbar_kws={"label": "MAE"},
)
ax.set_title("Mean MAE by Model x Fold (Phase 1, All 5 Models)",
             fontweight="bold", fontsize=13)
ax.set_xlabel("Fold")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## 6. Ranking Models theo trung binh MASE (Phase 1)

In [ ]:
# xep hang model theo MASE trung binh (Phase 1)
# MASE < 1 nghia la tot hon Seasonal Naive baseline
df_p1 = df[df["phase"] == 1].copy()

ranking = (
    df_p1.groupby("model")[METRIC_COLS]
    .agg(["mean", "std"])
    .round(4)
)
# flatten column names
ranking.columns = [f"{col}_{stat}" for col, stat in ranking.columns]
ranking = ranking.sort_values("MASE_mean")

print("=" * 70)
print("  RANKING MODELS THEO MASE (Phase 1) - THAP LA TOT")
print("=" * 70)
display(ranking)

## 7. Ket luan

Tu cac bieu do tren, co the rut ra:
- **LSTM** nhin chung co MAE va MASE thap nhat va on dinh nhat
- **Prophet** la model thu 5, co the so sanh truc tiep voi SARIMAX
- **MASE < 1.0** cho thay model du doan tot hon Seasonal Naive baseline
- **Phase 2** (enhanced features) co the cai thien hoac khong, tuy vao model

> *Ghi chu: ket qua tren dung MOCK data. Ket qua tren REAL data se khac.*